In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
test_passenger_id = test['PassengerId'].copy()

train['is_train'] = 1
test['is_train'] = 0
test['Survived'] = np.nan

#对test和train进行合并
combined = pd.concat([train, test], axis=0, ignore_index=True)
print(combined.shape)

In [ ]:
combined['Title'] = combined['Name'].str.extract(r',\s*([^\.]*)\.')

print(combined['Title'].value_counts())

# 把少见的称呼统一成为Rare
title_mapping = {
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs',
    'Lady': 'Rare',
    'Countess': 'Rare',
    'Capt': 'Rare',
    'Col': 'Rare',
    'Don': 'Rare',
    'Dr': 'Rare',
    'Major': 'Rare',
    'Rev': 'Rare',
    'Sir': 'Rare',
    'Jonkheer': 'Rare',
    'Dona': 'Rare'
}
combined['Title'] = combined['Title'].replace(title_mapping)

print(combined['Title'].value_counts())

In [ ]:
#对Age进行缺失填补
print("填补前Age缺失数:", combined['Age'].isnull().sum())

combined['Age'] = combined.groupby('Title')['Age'].transform(
    lambda x: x.fillna(x.median())
)

print("填补后Age缺失数:", combined['Age'].isnull().sum())

In [ ]:
#对Embarked进行缺失填补
print("填补前Embarked缺失数:", combined['Embarked'].isnull().sum())

combined['Embarked'] = combined['Embarked'].fillna(combined['Embarked'].mode()[0])

print("填补后Embarked缺失数:", combined['Embarked'].isnull().sum())

In [ ]:
#对Fare进行缺失填补
print("填补前Fare缺失数:", combined['Fare'].isnull().sum())

combined['Fare'] = combined.groupby('Pclass')['Fare'].transform(
    lambda x: x.fillna(x.median())
)

print("填补后Fare缺失数:", combined['Fare'].isnull().sum())

In [ ]:
#对各种类别的变量进行分类
combined['HasCabin'] = combined['Cabin'].notnull().astype(int)
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1
combined['IsAlone'] = (combined['FamilySize'] == 1).astype(int)
combined['Sex'] = combined['Sex'].map({'male': 0, 'female': 1})
combined = pd.get_dummies(combined, columns=['Embarked'], prefix='Embarked')
combined = pd.get_dummies(combined, columns=['Title'], prefix='Title')
combined.head()

In [ ]:
#丢弃不需要的变量类别，并检查缺失值
drop_cols = ['Name', 'Ticket', 'Cabin']
combined = combined.drop(columns=drop_cols)

print(combined.columns.tolist())

print(combined.isnull().sum()[combined.isnull().sum() > 0])

In [28]:
train_processed = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_processed = combined[combined['is_train'] == 0].drop(columns=['is_train', 'Survived'])

print("Train processed shape:", train_processed.shape)
print("Test processed shape:", test_processed.shape)

# 保存文件
train_processed.to_csv('data/processed/train_processed.csv', index=False)
test_processed.to_csv('data/processed/test_processed.csv', index=False)

print("已保存到 data/processed/")

Train processed shape: (891, 20)
Test processed shape: (418, 19)
已保存到 data/processed/
